# Notebook 1 — Naive RAG
**Livro:** Os Sertões — Euclides da Cunha  
**Estratégia:** Chunking fixo → embeddings → busca por similaridade → geração com Claude

## 1. Instalação de dependências

In [ ]:
!pip install -q anthropic pypdf chromadb sentence-transformers langchain langchain-community tiktoken

## 2. Importações e configuração da API

In [ ]:
import os
import re
import anthropic
import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from typing import List

# Configure sua chave da API Anthropic
# Opção 1: variável de ambiente (recomendado)
# export ANTHROPIC_API_KEY="sk-ant-..."
# Opção 2: direto no código (não commitar!)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
print("Cliente Anthropic inicializado com sucesso.")

## 3. Download e leitura do PDF

In [ ]:
import urllib.request

PDF_URL = "https://fundar.org.br/wp-content/uploads/2021/06/os-sertoes.pdf"
PDF_PATH = "os-sertoes.pdf"

if not os.path.exists(PDF_PATH):
    print("Baixando o PDF...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print("Download concluído.")
else:
    print("PDF já existe localmente.")

def extract_text_from_pdf(path: str) -> str:
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

full_text = extract_text_from_pdf(PDF_PATH)
print(f"Total de caracteres extraídos: {len(full_text):,}")

## 4. Chunking fixo (Naive RAG)

In [ ]:
def naive_chunk(text: str, chunk_size: int = 1000, overlap: int = 200) -> List[str]:
    """Divide o texto em chunks de tamanho fixo com sobreposição."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Limpeza básica do texto
clean_text = re.sub(r'\s+', ' ', full_text).strip()
chunks = naive_chunk(clean_text, chunk_size=1000, overlap=200)

print(f"Total de chunks gerados: {len(chunks)}")
print(f"\nExemplo do chunk 0:\n{chunks[0][:300]}...")

## 5. Geração de embeddings e indexação no ChromaDB

In [ ]:
# Modelo de embedding multilíngue (suporta português)
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Inicializa o banco vetorial ChromaDB em memória
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="naive_rag_sertoes",
    metadata={"hnsw:space": "cosine"}
)

# Gera embeddings e insere em lotes
BATCH_SIZE = 100
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    embeddings = embed_model.encode(batch, show_progress_bar=False).tolist()
    ids = [f"chunk_{j}" for j in range(i, i + len(batch))]
    collection.add(documents=batch, embeddings=embeddings, ids=ids)
    if i % 500 == 0:
        print(f"  Indexados {i + len(batch)}/{len(chunks)} chunks...")

print(f"\nIndexação concluída! Total no banco: {collection.count()} documentos.")

## 6. Função de busca e geração (RAG pipeline)

In [ ]:
def retrieve(query: str, k: int = 5) -> List[str]:
    """Recupera os k chunks mais relevantes para a query."""
    query_embedding = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )
    return results["documents"][0]

def generate_answer(query: str, context_chunks: List[str]) -> str:
    """Gera resposta usando Claude com os chunks como contexto."""
    context = "\n\n---\n\n".join(context_chunks)
    
    prompt = f"""Você é um assistente especializado em literatura brasileira.
Use APENAS o contexto abaixo para responder à pergunta.
Se a informação não estiver no contexto, diga que não encontrou.

CONTEXTO:
{context}

PERGUNTA: {query}

RESPOSTA:"""
    
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

def naive_rag(query: str, k: int = 5) -> dict:
    """Pipeline completo: retrieve → generate."""
    chunks_retrieved = retrieve(query, k=k)
    answer = generate_answer(query, chunks_retrieved)
    return {"query": query, "context": chunks_retrieved, "answer": answer}

print("Pipeline Naive RAG pronto!")

## 7. Respondendo às 5 questões

In [ ]:
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em Os Sertões? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

results = []
for i, q in enumerate(questions, 1):
    print(f"\n{'='*70}")
    print(f"QUESTÃO {i}: {q}")
    print('='*70)
    result = naive_rag(q, k=5)
    results.append(result)
    print(result["answer"])

## 8. Salvando resultados

In [ ]:
import json

with open("resultados_naive_rag.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Resultados salvos em resultados_naive_rag.json")